# Module 4 — Entity and Relationship Extraction with LLMs

**The gap, from Modules 1-3:** the graph so far holds what's *structured* — filings chunked into
text, plus executives/financials/news pulled from external sources. But the filing text itself is
full of entities and relationships that never get surfaced: named regulations, subsidiaries,
products, risk factors, all sitting inertly inside `Chunk.text`, invisible to Cypher.

**What we build in this module:**
- Mine chunk text with an LLM, and see why prompt design matters before committing to an
  approach: a bare "extract entities and relationships" prompt produces a different,
  inconsistent type vocabulary almost every run — not usable for a Cypher query that expects a
  `SUBSIDIARY_OF` edge to always be spelled `SUBSIDIARY_OF`
- Add a predefined type schema to fix the vocabulary problem, then confront the subtler issue it
  doesn't fix: asked to extract entities and relationships in the same breath, the model invents
  relationships to things it never firmly committed to as entities
- Split the process — settle the entity list first, double-check it with a reflection pass, *then*
  extract relationships constrained to that settled list — removing that class of hallucination
  by construction
- Ship that two-phase pipeline to `src/extraction/` and run it for real, writing generic
  `RecognisedEntity` nodes and `RELATED_TO` edges to Neo4j — deliberately not merged across
  chunks/documents or reconciled with the curated `Company`/`Person` nodes from Modules 1/3;
  that reconciliation is Module 5's job
- Add two new agent tools that query this extracted structure directly, and compare an agent
  with and without them on questions that need it

**New components introduced:**
- `extraction.validators` — `EntityType`/`RelationshipType` enums, `ExtractedEntity`/`ExtractedRelationship`
- `extraction.prompts` — role/scope preamble, entity/reflection/relationship prompts
- `extraction.entities` — `extract_entities()`: pass A (entities) + pass B (reflection)
- `extraction.relationships` — `extract_relationships()`: pass C (relationships, closed entity list); `write_extraction_to_graph()`
- `ingestion.schema.apply_extraction_schema()` — `RecognisedEntity(string, doc_id)` uniqueness constraint
- `retrieval.graph_nav.get_recognised_entities`/`get_entity_relationships` + matching `agent.tools` — query the extracted structure directly; bound as `MODULE_4_TOOLS` with `MODULE_4_STRATEGY_PROMPT`

> Run `scripts/run_extraction.py` to process the full corpus (long-running; this notebook only
> runs a small demo batch).

## 1. Picking chunks to work with

Not every chunk is worth extracting from — plenty of 10-K text is boilerplate (signature blocks,
certifications) with little to mine. We'll work with one chunk throughout the naive → rigorous →
two-phase comparison below: 3M's **Item 1. Business** overview, which is short enough to read in
full but already has a company, a jurisdiction, a regulator, a named law, and three business
segments packed into a few sentences — enough variety to see a prompt succeed or fail on more
than one entity type at once.

In [1]:
from langchain_core.documents import Document
from financial_advisor.services.neo4j_service import neo4j_service

BUSINESS_CHUNK_ID = "3M/3M_2025_10K.pdf/6"

row = neo4j_service.run_query(
    "MATCH (c:Chunk {id: $id}) RETURN c.text AS text", {"id": BUSINESS_CHUNK_ID}
)[0]
business_doc = Document(page_content=row["text"], metadata={"id": BUSINESS_CHUNK_ID})
print(business_doc.page_content)


Item 1. Business
3M Company was incorporated in 1929 under the laws of the State of Delaware to continue operations begun in 1902. The Company's ticker symbol is MMM. As used herein, the term '3M' or 'Company' includes 3M Company and its subsidiaries unless the context indicates otherwise. In this document, for any references to Note 1 through Note 20, refer to the Notes to Consolidated Financial Statements in Item 8.
Available Information : The Securities and Exchange Commission (SEC) maintains a website that contains reports, proxy and information statements, and other information regarding issuers, including the Company, that file electronically with the SEC. The public can obtain any documents that the Company files with the SEC at https://www.sec.gov. The Company files annual reports, quarterly reports, proxy statements and other documents with the SEC under the Securities Exchange Act of 1934 (Exchange Act).
3M also makes available free of charge through its website (https://inve

## 2. A naive prompt

The most obvious approach: ask the model to "extract entities and relationships," give it a
loose schema (a free-text `type` field, no constraints), and see what comes back. No role, no
scope, no predefined vocabulary — just the instruction and the text.

In [2]:
# Ad hoc, throwaway schema — deliberately NOT the one in extraction.validators. This whole cell
# exists to be visibly worse than what ships in src/, so it stays notebook-only (see adr/0007).
from pydantic import BaseModel, Field

from financial_advisor.clients import get_llm


class NaiveEntity(BaseModel):
    name: str
    type: str
    description: str | None = None


class NaiveRelationship(BaseModel):
    source: str
    target: str
    type: str


class NaiveResult(BaseModel):
    entities: list[NaiveEntity] = Field(default_factory=list)
    relationships: list[NaiveRelationship] = Field(default_factory=list)


naive_prompt = (
    "Extract entities and relationships from this text, focusing on things relevant to "
    f"financial analysis:\n\n{business_doc.page_content}"
)
naive_result = get_llm().with_structured_output(NaiveResult).invoke(naive_prompt)

for e in naive_result.entities:
    print(f"{e.type:25s} | {e.name}")
print()
for r in naive_result.relationships:
    print(f"{r.source} -[{r.type}]-> {r.target}")

Company                   | 3M Company
Company                   | 3M
Regulatory Agency         | SEC
Law                       | Securities Exchange Act of 1934
Ticker Symbol             | MMM
Website                   | investors.3M.com
Business Segment          | Safety and Industrial
Business Segment          | Transportation and Electronics
Business Segment          | Consumer
Geographic Scope          | Global Presence

3M Company -[has ticker symbol]-> MMM
3M -[includes]-> 3M Company
3M -[files reports with]-> SEC
3M -[files under]-> Securities Exchange Act of 1934
SEC -[maintains website]-> https://www.sec.gov
3M -[provides reports through]-> investors.3M.com
3M -[operates in]-> Safety and Industrial
3M -[operates in]-> Transportation and Electronics
3M -[operates in]-> Consumer
3M -[has]-> Global Presence


**What went wrong.** Run this cell and look at the `type` column: this run came back with
`Company`, `Regulatory Agency`, `Law`, `Ticker Symbol`, `Website`, `Business Segment`, and
`Geographic Scope` — seven ad hoc types for ten entities, none of them chosen from any fixed
vocabulary because none was given. Run the cell again and you'll likely get a *different* set of
labels — earlier runs of this same cell came back with `Jurisdiction`, `SEC Filing`, `Filing
Type`, `Subsidiaries`, and `Competitors (general)` instead, none of which show up here. That's
fatal for a graph you intend to query: `MATCH (n:RecognisedEntity {type: "Regulation"})` only
works if "Regulation" is spelled the same way every time it's produced, and nothing here enforces
that.

The relationships show the same problem from a different angle. This run produced 10 edges over
eight distinct verb phrasings — `has ticker symbol`, `includes`, `files reports with`, `files
under`, `maintains website`, `provides reports through`, `operates in` (×3), and `has` — a
different invented verb for nearly every kind of fact. Instructing the model to "focus on things
relevant to financial analysis" did not stop it from extracting the `https://www.sec.gov` URL as
a relationship target either, or from inventing a `Website` type for `investors.3M.com` — an
underspecified schema gives the model no way to know what counts as an entity worth naming versus
what belongs, if anywhere, in a relationship's evidence text.

## 3. Adding Rigor: Role, Scope, Predefined Types

Second attempt: give the model a role ("financial analyst"), a scope (this is a 10-K, extract
only what's explicitly stated), and a closed type vocabulary for both entities and
relationships — the actual `EntityType`/`RelationshipType` enums from `extraction.validators`.
Still one call for entities *and* relationships together, same as the naive version — only the
type discipline changes.

In [3]:
from financial_advisor.extraction.validators import EntityType, RelationshipType


class SingleShotEntity(BaseModel):
    string: str
    type: EntityType


class SingleShotRelationship(BaseModel):
    source: str
    target: str
    type: RelationshipType


class SingleShotResult(BaseModel):
    entities: list[SingleShotEntity] = Field(default_factory=list)
    relationships: list[SingleShotRelationship] = Field(default_factory=list)


entity_types = ", ".join(t.value for t in EntityType)
relationship_types = ", ".join(t.value for t in RelationshipType)
single_shot_prompt = f"""\
You are a financial analyst extracting structured knowledge from a company's SEC 10-K filing.
Extract entities of these types only: {entity_types}.
Extract relationships of these types only: {relationship_types}. Source/target must be entities
you extracted. Only extract what the text explicitly states.

Text:
{business_doc.page_content}"""

single_shot_result = get_llm().with_structured_output(SingleShotResult).invoke(single_shot_prompt)

for e in single_shot_result.entities:
    print(f"{e.type.value:20s} | {e.string}")
print()
for r in single_shot_result.relationships:
    print(f"{r.source} -[{r.type.value}]-> {r.target}")

Company              | 3M Company
Company              | 3M
Company              | SEC
Company              | Securities and Exchange Commission
Regulation           | Securities Exchange Act of 1934
Filing               | Annual Report on Form 10-K
Filing               | Quarterly Reports on Form 10-Q
Filing               | Current Reports on Form 8-K
Product              | Safety and Industrial
Product              | Transportation and Electronics
Product              | Consumer
Location             | global presence
Risk                 | competition

3M -[OPERATES_IN]-> global presence
3M -[PRODUCES]-> Safety and Industrial
3M -[PRODUCES]-> Transportation and Electronics
3M -[PRODUCES]-> Consumer
3M -[EXPOSED_TO]-> competition
3M Company -[REGULATED_BY]-> Securities Exchange Act of 1934


**Better, but not fixed.** The type vocabulary is now consistent — every entity and relationship
is spelled from the fixed enum, run after run. The filing-type gap this section used to
illustrate is gone now that `EntityType` includes `Filing`: "Annual Report on Form 10-K,"
"Quarterly Reports on Form 10-Q," and "Current Reports on Form 8-K" all land under `Filing`, not
`FinancialMetric`. But a different failure mode still shows up. Look at the entity list: the
regulator itself — "SEC" and "Securities and Exchange Commission" — both still come back typed
`Company`, not `Regulation`. `EntityType` still has no type for a regulatory *body* (only
`Regulation`, for the law or rule itself, and now `Filing`, for the document), so `Company`
remains the nearest fit for any named organization, real business or not. A second, smaller
miss: "global presence" — a vague noun phrase from "3M is a diversified technology company with
a global presence" — gets typed `Location`, standing in for an actual place. And this run again
dropped the `3M -[SUBSIDIARY_OF]-> 3M Company` link the naive prompt in section 2 caught by
accident — a completeness miss, not a mislabeling. Fixing one vocabulary gap (`Filing`) closed
one specific failure mode cleanly, but the type discipline still can't stop the model from
reaching for the closest available label when the right one doesn't exist, or guarantee the
extraction stays complete.

## 4. Splitting the Process: Entities → Reflection → Relationships

Third attempt, and the one that ships to `src/extraction/`: stop asking for entities and
relationships in one call. Settle the entity list first — a first pass, then a second
"anything missed?" reflection pass shown the text plus its own first-pass list — and only once
that list is settled, extract relationships constrained to name only entities from it. This is
`extract_entities()` and `extract_relationships()` from `extraction.entities`/
`extraction.relationships`, not a notebook throwaway — the real implementation.

In [4]:
from financial_advisor.extraction.entities import extract_entities

entity_result = extract_entities(business_doc)
for e in entity_result.entities:
    print(f"{e.type.value:20s} | {e.string}")


[extract-entities] pass A: 14 entit(y/ies) found


[extract-entities] reflection: 3 addition(s)/correction(s)
Company              | 3M Company
Company              | 3M
Company              | Company
Company              | Securities and Exchange Commission
Company              | SEC
Regulation           | Exchange Act
Regulation           | Securities Exchange Act of 1934
Filing               | Form 10-K
Filing               | Form 10-Q
Filing               | Form 8-K
Location             | State of Delaware
Product              | Safety and Industrial
Product              | Transportation and Electronics
Product              | Consumer
Company              | the Company
Filing               | Annual Report on Form 10-K
Filing               | Current Reports on Form 8-K


Watch the `[extract-entities]` log lines above: pass A found 14 entities this run, reflection
added 3 more — 17 total, spanning `Company`, `Regulation`, `Filing`, `Location`, and `Product`.
The near-duplicate-entity problem from Module 5's backlog is still visible — "3M Company"/"3M"/
"the Company" all stay separate `Company` entities, and "Securities and Exchange Commission"/
"SEC" stay separate too — but at least they're now consistently typed rather than split across
two different types for what's obviously the same organization, as an earlier run of this same
cell once did.

The filing-type vocabulary gap this section used to illustrate is gone: with `EntityType.FILING`
added, "Form 10-K," "Form 10-Q," "Form 8-K," and their longer forms ("Annual Report on Form
10-K," "Current Reports on Form 8-K") all land under `Filing` now, not force-fit into
`FinancialMetric`. Notice, though, that fixing the *type* didn't fix a separate, pre-existing
*duplication* problem: "Form 10-K" and "Annual Report on Form 10-K" are still two different
`Filing` entities for what's the same document mentioned two different ways in the text — the
vocabulary gap and the near-duplicate-naming problem are separate issues, and only the first one
was in scope here. `EntityType` will likely need further refinement as more filing text runs
through it — this run's own single-shot cell above still force-fits the regulator itself (`SEC`)
into `Company` for lack of a `Regulator`/`Agency` type, the next gap worth closing.

In [5]:
from financial_advisor.extraction.relationships import extract_relationships

known_entity_names = sorted({e.resolved_string for e in entity_result.entities})
relationship_result = extract_relationships(business_doc, known_entity_names)
for r in relationship_result.relationships:
    print(f"{r.source} -[{r.type.value}]-> {r.target}")


[extract-relationships] dropped 1 relationship(s) referencing unknown entities
[extract-relationships] 7 relationship(s) found
3M Company -[REGULATED_BY]-> State of Delaware
3M -[OPERATES_IN]-> Safety and Industrial
3M -[OPERATES_IN]-> Transportation and Electronics
3M -[OPERATES_IN]-> Consumer
3M -[OPERATES_IN]-> SEC
Company -[OPERATES_IN]-> SEC
Company -[REGULATED_BY]-> Exchange Act


Notice the difference from the naive and single-shot attempts above: nothing here points at an
entity that was never confirmed to exist — and this run shows the backstop actually doing that
job, not just in theory. The log line above, `dropped 1 relationship(s) referencing unknown
entities`, is `_filter_valid_relationships` (`extraction/relationships.py`) catching the model
naming something outside the settled list anyway, despite the prompt constraint, and dropping it
rather than writing it to the graph. Compare that to section 2's naive relationships, which
included `3M -[includes]-> 3M Company subsidiaries` — a target that was never extracted as its
own entity and had nothing there to catch it. `extract_relationships` structurally can't
hallucinate a target the same way: it only ever offers the model names already confirmed to
exist in the settled entity list, so whatever slips through anyway gets filtered, logged, and
dropped instead of silently written. Splitting entities from relationships didn't just make the
output cleaner — it removed an entire class of hallucination by construction, and gave that
removal an audit trail.

## 4a. A Refinement: Resolving Coreference

Look back at the very first chunk in section 1 — its own text defines the term this filing will
use for itself: *"As used herein, the term '3M' or 'Company' includes 3M Company and its
subsidiaries unless the context indicates otherwise."* From that point on, much of the filing
refers to 3M not by name but as **"the Company"** — a defined term, not a generic pronoun, so it
reads as a legitimate named entity to extract.

Extracted literally, every one of those mentions becomes its own entity, disconnected from the
one named "3M". Without this section's fix, a run over the Risk Factors chunk below can produce
output like this:

```
(Company) The Company -[EXPOSED_TO]-> (Risk) AFFF multi-district litigation
(Company) The Company -[EXPOSED_TO]-> (Risk) PWS Settlement
(Company) 3M -[EXPOSED_TO]-> (FinancialMetric) $10.5 billion to $12.5 billion in total
```

Same company, same filing, same kind of relationship — split across two unconnected identities
just because one mention happened to be phrased as a name and another as a defined term.

**Why this isn't Module 5's job.** Module 5 (similarity/entity resolution, coming up) reconciles
extracted mentions by *name similarity* — fuzzy string matching narrows candidates before an LLM
judges whether they're the same thing. "the Company" and "3M" share no common substring, so "3M"
would never even become a candidate for "the Company" to be judged against — the pair never
reaches the LLM judge at all. Module 5 is the right tool for genuinely different name strings
("Solventum Corporation" vs "Solventum"); it structurally can't catch a defined-term
self-reference.

**Where coreference gets resolved.** Reflection (pass B) already receives the settled
first-pass entity list *and* the source text side by side — exactly the two things needed to spot
*any* coreference, not just a filer referring to itself as "the Company". `ExtractedEntity`
keeps two fields — `string` (the literal text, e.g. "the Company", never discarded) and
`resolved_string` (the canonical entity this mention refers to) — and resolving them is
reflection's job: given the entities already found and the chunk text as evidence, check whether
any entity is really just another way of referring to a *different* entity already in the list,
and if so, report `resolved_string` as that other entity's own `string`. This generalizes well
beyond the "the Company" case — the same mechanism can resolve abbreviations to their spelled-out
forms too, for example:

```
Company     | string='The Company' -> resolved_string='3M'
Location    | string='U.S.' -> resolved_string='United States'
Regulation  | string='CERCLA' -> resolved_string="United States Comprehensive Environmental
              Response, Compensation and Liability Act of 1980 ('CERCLA')"
Regulation  | string='EPA' -> resolved_string='U.S. Environmental Protection Agency (EPA)'
```

**The backstop.** A natural-language instruction is never 100% reliable — this notebook has
already made that point twice (sections 2 and 3) — so `resolved_string` still gets checked in
code, not just asked for nicely, the same way `_filter_valid_relationships` already checks
relationships. `extraction.entities._restrict_resolved_strings_to_settled_entities` resets
`resolved_string` back to `string` whenever it doesn't match another entity's `string` that's
actually in the settled list *and* of the same type — catching both an invented canonical form
nobody extracted, and a cross-type mix-up (a `Risk` sentence merely mentioning "the Company"
getting redirected to a `Company` entity's string).

**A trade-off worth knowing about.** Resolving coreference this way can only resolve a mention to
an entity that's actually extracted somewhere in the *same* chunk; if "3M" itself isn't a settled
entity in a given chunk, "the Company" there has nothing to resolve to and stays
self-referential. In practice a chunk that talks about "the Company" at length also names the
company directly at least once, so this is expected to be rare.

In [6]:
from financial_advisor.extraction.entities import extract_entities
from financial_advisor.extraction.relationships import extract_relationships

COREF_CHUNK_ID = "3M/3M_2025_10K.pdf/17"  # Risk Factors — Legal and Regulatory Proceedings (PFAS)

row = neo4j_service.run_query(
    "MATCH (c:Chunk {id: $id}) RETURN c.text AS text", {"id": COREF_CHUNK_ID}
)[0]
coref_doc = Document(page_content=row["text"], metadata={"id": COREF_CHUNK_ID})

coref_entities = extract_entities(coref_doc).entities
print("Entities where the literal text differs from the resolved identity:")
for e in coref_entities:
    if e.string != e.resolved_string:
        print(f"  {e.type.value:10s} | string={e.string!r:20s} -> resolved_string={e.resolved_string!r}")

known_coref_entities = sorted({e.resolved_string for e in coref_entities})
coref_relationships = extract_relationships(coref_doc, known_coref_entities).relationships
print("\nRelationships (source/target are now the resolved identity):")
for r in coref_relationships:
    print(f"  {r.source} -[{r.type.value}]-> {r.target}")


[extract-entities] pass A: 32 entit(y/ies) found


[extract-entities] reflection: 7 addition(s)/correction(s)
Entities where the literal text differs from the resolved identity:


[extract-relationships] dropped 2 relationship(s) referencing unknown entities
[extract-relationships] 25 relationship(s) found

Relationships (source/target are now the resolved identity):
  The Company -[EXPOSED_TO]-> PFAS
  3M -[EXPOSED_TO]-> PFAS
  PFAS -[SUPPLIES]-> commercial aircraft
  PFAS -[SUPPLIES]-> low-emissions vehicles
  PFAS -[SUPPLIES]-> medical products
  PFAS -[SUPPLIES]-> Aqueous Film Forming Foam (AFFF)
  PFAS -[SUPPLIES]-> surgical gowns and drapes
  PFAS -[SUPPLIES]-> printed circuit boards
  PFAS -[SUPPLIES]-> lithium ion batteries
  PFAS -[SUPPLIES]-> certain seals and gaskets
  3M -[PRODUCES]-> PFAS
  3M -[EXPOSED_TO]-> PFAS
  Solventum -[EXPOSED_TO]-> PFAS
  The Company -[REGULATED_BY]-> CERCLA
  The Company -[REGULATED_BY]-> United States Comprehensive Environmental Response, Compensation and Liability Act of 1980
  The Company -[REGULATED_BY]-> EPA
  The Company -[REGULATED_BY]-> United States Environmental Protection Agency
  3M -[CUSTOMER_OF]-> PWS Settle

**What this run actually shows.** Look at the loop above: it prints its header and then nothing —
every entity in `coref_entities` came back with `string == resolved_string`, including "The
Company" and "3M", which the text above establishes refer to the same filer. Reflection didn't
make the correction this time; the relationships below show it too, "The Company" and "3M"
staying two separate sources across near-identical facts (`The Company -[EXPOSED_TO]-> PFAS` and
`3M -[EXPOSED_TO]-> PFAS` both appear, rather than collapsing onto one). This isn't a one-off —
it's held across three separate runs of this cell this session, so treat it as this model's
actual behavior on this prompt rather than a fluke. This is the "backstop" paragraph above
playing out for real, just one step earlier than the backstop itself:
`_restrict_resolved_strings_to_settled_entities` only ever undoes a *wrong* resolution — it has
nothing to add when reflection simply never attempts one. A natural-language instruction ("check
whether any entity is really just another way of referring to a different entity") is a request,
not a guarantee, and this is a clean, repeatable example of it going unmet rather than going
wrong. The design is still the right one — coreference resolution has to happen somewhere, and
reflection is the only pass with both the settled entity list and the source text to resolve
against — it just isn't reliable enough on its own to promise it catches a given case, on this
model or any other.

## 5. Storage: `RecognisedEntity` and `RELATED_TO`

Two more decisions:

- **One generic node label, `RecognisedEntity`**, not per-type labels like `:Company`/`:Person`.
  Real labels would collide with the *curated* `Company`/`Person` nodes Modules 1/3 already
  populate — an extracted mention isn't confirmed to be the same node as the curated one yet
  (that reconciliation is Module 5). `type` lives as a property instead. The uniqueness key is
  `(string, doc_id)` — same string mentioned twice in the same document merges into one node;
  the same string in a different document, or a near-miss like "3M" vs. "3M Company," stays
  separate. No cross-document merging yet, by design.
- **One generic edge, `RELATED_TO {type, evidence, chunk_id}`**, not dynamic edge types matching
  the relationship enum. Same collision reasoning — a `RecognisedEntity-[:SUBSIDIARY_OF]->
  RecognisedEntity` edge would sit right next to the *real* curated `Company-[:SUBSIDIARY_OF]->
  Company` edges from Module 3's Wikidata structure data, and it's not obvious which is which
  without checking the node labels. It also means Module 5's relationship reconciliation scans
  one edge type instead of nine.

Provenance is `(:RecognisedEntity)-[:MENTIONED_IN]->(:Chunk)`, so every extracted node traces back
to the exact text it came from. `string` on a `RecognisedEntity` is the *resolved* identity (see
4a above) — the `(string, doc_id)` uniqueness key is applied after coreference resolution, not
before, so "the Company" and "3M" mentions already collapse onto one node by the time this key
matters. Every distinct literal form actually seen lands in a separate `mentions` list property.

In [7]:
from financial_advisor.ingestion.schema import apply_extraction_schema

apply_extraction_schema()

  [schema] OK  recognised_entity_key
[schema] 1/1 statements applied — all good


## 6. Running the Pipeline for Real

A small batch — the Business Overview chunk from above plus three more picked for variety: a
dense Legal/Regulatory Risk Factors passage (PFAS litigation — lots of named substances and
dollar figures), the MD&A Overview (names 3M's 2024 spin-off of Solventum Corporation and a
financial-highlights table), and a subsidiaries table from the 2024 filing (clean, unambiguous
`SUBSIDIARY_OF`-shaped data — every row is `(company name, jurisdiction)`). Same
`extract_entities` → `extract_relationships` → `write_extraction_to_graph` pipeline as above, now
actually writing to Neo4j. The full corpus is `scripts/run_extraction.py`'s job, not this
notebook's — this is a few chunks so the result is visible in one run.

In [8]:
from financial_advisor.extraction.relationships import write_extraction_to_graph

DEMO_CHUNK_IDS = [
    BUSINESS_CHUNK_ID,
    "3M/3M_2025_10K.pdf/17",  # Risk Factors — Legal and Regulatory Proceedings (PFAS)
    "3M/3M_2025_10K.pdf/33",  # MD&A Overview (Solventum separation, financial highlights)
    "3M/3M_2024_10K.pdf/244",  # 3M Company and Consolidated Subsidiaries
]

rows = neo4j_service.run_query(
    "MATCH (c:Chunk) WHERE c.id IN $ids RETURN c.id AS id, c.text AS text, c.doc_id AS doc_id",
    {"ids": DEMO_CHUNK_IDS},
)

for i, row in enumerate(rows, start=1):
    print(f"[{i}/{len(rows)}] chunk {row['id']}")
    doc = Document(page_content=row["text"], metadata={"id": row["id"]})

    entities = extract_entities(doc).entities
    known_entities = sorted({e.resolved_string for e in entities})
    relationships = extract_relationships(doc, known_entities).relationships
    write_extraction_to_graph(entities, relationships, chunk_id=row["id"], doc_id=row["doc_id"])
    neo4j_service.run_query("MATCH (c:Chunk {id: $id}) SET c.extracted = true", {"id": row["id"]})
    print(f"    wrote {len(entities)} entities, {len(relationships)} relationships")


[1/4] chunk 3M/3M_2025_10K.pdf/6


[extract-entities] pass A: 18 entit(y/ies) found


[extract-entities] reflection: 3 addition(s)/correction(s)


[extract-relationships] 10 relationship(s) found
    wrote 18 entities, 10 relationships
[2/4] chunk 3M/3M_2025_10K.pdf/17


[extract-entities] pass A: 24 entit(y/ies) found


...

[extract-entities] reflection: 11 addition(s)/correction(s)


[extract-relationships] 22 relationship(s) found
    wrote 35 entities, 22 relationships
[4/4] chunk 3M/3M_2024_10K.pdf/244


[extract-entities] pass A: 54 entit(y/ies) found


[extract-entities] reflection: 1 addition(s)/correction(s)


[extract-relationships] 35 relationship(s) found
    wrote 55 entities, 35 relationships


In [9]:
rows = neo4j_service.run_query(
    """
    MATCH (a:RecognisedEntity)-[r:RELATED_TO]->(b:RecognisedEntity)
    RETURN a.string AS source, a.type AS source_type, r.type AS relationship,
           b.string AS target, b.type AS target_type
    ORDER BY relationship
    LIMIT 25
    """
)
for row in rows:
    print(f"({row['source_type']}) {row['source']} -[{row['relationship']}]-> "
          f"({row['target_type']}) {row['target']}")

(Company) The Company -[EXPOSED_TO]-> (Product) perfluoroalkyl and polyfluoroalkyl substances
(Company) The Company -[EXPOSED_TO]-> (Product) fluorochemicals
(Company) The Company -[EXPOSED_TO]-> (Company) U.S. and foreign regulatory authorities
(FinancialMetric) Operating income margin -[EXPOSED_TO]-> (Regulation) 2025 PFAS-related New Jersey Settlement
(FinancialMetric) Earning per diluted share (EPS) -[EXPOSED_TO]-> (Filing) the Income from Unconsolidated Subsidiaries, Net of Taxes discussion below
(FinancialMetric) Earning per diluted share (EPS) -[EXPOSED_TO]-> (FinancialMetric) pension expense
(FinancialMetric) Operating income margin -[EXPOSED_TO]-> (Risk) special items
(FinancialMetric) Earning per diluted share (EPS) -[EXPOSED_TO]-> (FinancialMetric) nonoperating net interest expense
(FinancialMetric) Operating income margin -[EXPOSED_TO]-> (Risk) cost dis-synergies
(FinancialMetric) Operating income margin -[EXPOSED_TO]-> (Filing) Note 17
(FinancialMetric) Earning per diluted

The subsidiaries table is worth checking on its own — it's the chunk with the clearest ground
truth (every row *should* produce one `Company -[SUBSIDIARY_OF]-> Company` edge with a
`Location`-typed jurisdiction alongside it), so it's the easiest one to sanity-check the pipeline
against.

In [10]:
rows = neo4j_service.run_query(
    """
    MATCH (e:RecognisedEntity)-[:MENTIONED_IN]->(:Chunk {id: "3M/3M_2024_10K.pdf/244"})
    OPTIONAL MATCH (e)-[r:RELATED_TO {type: "SUBSIDIARY_OF"}]->(parent:RecognisedEntity)
    RETURN e.string AS entity, e.type AS type, parent.string AS parent_of
    ORDER BY type, entity
    """
)
for row in rows:
    print(f"{row['type']:10s} | {row['entity']:45s} | SUBSIDIARY_OF -> {row['parent_of']}")

Company    | 3M Belgium BV                                 | SUBSIDIARY_OF -> 3M Company
Company    | 3M Canada Company - Compagnie 3M Canada       | SUBSIDIARY_OF -> 3M Company
Company    | 3M Chemical Operations LLC                    | SUBSIDIARY_OF -> 3M Company
Company    | 3M China Limited                              | SUBSIDIARY_OF -> 3M Company
Company    | 3M Company                                    | SUBSIDIARY_OF -> None
Company    | 3M Deutschland GmbH                           | SUBSIDIARY_OF -> 3M Company
Company    | 3M EMEA GmbH                                  | SUBSIDIARY_OF -> 3M Company
Company    | 3M Fall Protection Company                    | SUBSIDIARY_OF -> 3M Company
Company    | 3M Financial Management Company               | SUBSIDIARY_OF -> 3M Company
Company    | 3M Foreign Holding LLC                        | SUBSIDIARY_OF -> 3M Company
Company    | 3M France S.A.S.                              | SUBSIDIARY_OF -> 3M Company
Company    | 3M Global Capi

### Check the database
Let's check our database using the browser at [http://localhost:7474/browser/](http://localhost:7474/browser/)

```
MATCH (n:RecognisedEntity {type: "Company"}) RETURN n LIMIT 25
```

## 7. New Questions This Structure Unlocks

Everything above justified the pipeline in the abstract. Here's the payoff: two new agent tools,
`get_recognised_entities`/`get_entity_relationships` (`retrieval/graph_nav.py`,
`agent/tools.py`), that query `RecognisedEntity`/`RELATED_TO` directly instead of asking the
model to re-derive structure from raw chunk text on every question. They're bound alongside
Module 3's tools as `MODULE_4_TOOLS`, with `MODULE_4_STRATEGY_PROMPT` telling the strategy agent
when to reach for them.

We'll ask the *same* two questions of two agents — one built with `MODULE_3_TOOLS` (no access to
the extracted structure, has to work from raw chunk text), one with `MODULE_4_TOOLS` — and
compare. Both questions are chosen to be hard specifically *because* the answer requires
aggregating over the full subsidiaries table, not because the underlying text is hidden or
missing.

In [11]:
from financial_advisor.agent.graph import build_agent
from financial_advisor.agent.prompts import MODULE_3_STRATEGY_PROMPT, MODULE_4_STRATEGY_PROMPT
from financial_advisor.agent.state import initial_state
from financial_advisor.agent.tools import MODULE_3_TOOLS, MODULE_4_TOOLS

agent_without = build_agent(MODULE_3_TOOLS, MODULE_3_STRATEGY_PROMPT)
agent_with = build_agent(MODULE_4_TOOLS, MODULE_4_STRATEGY_PROMPT)


def ask(agent, question: str) -> dict:
    result = agent.invoke(initial_state(question), {"recursion_limit": 50})
    tool_sequence = [entry["tool"] for entry in result["tool_call_log"]]
    print(f"[{result['retrieval_iterations']} retrieval round(s)] tools called: {tool_sequence}")
    print(f"\nA: {result['answer']}")
    return result

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

### 7a. "What subsidiaries are mentioned, and where?"

The table itself is fully inside one chunk (`3M/3M_2024_10K.pdf/244`), so a text search *can*
retrieve the right raw material in one shot. The question is what each agent does with it.

In [12]:
Q1 = (
    "What are the subsidiary companies mentioned in 3M's 2024 10-K (doc_id "
    "3M/3M_2024_10K.pdf), and in which countries/jurisdictions are they organized?"
)

print("===== WITHOUT the new tools =====")
result_without_q1 = ask(agent_without, Q1)

===== WITHOUT the new tools =====


[strategy] iteration 1: 1 tool call(s) planned
    - fulltext_search({'query': 'subsidiaries organized under the laws of', 'company_id': '3M', 'year': 2024, 'k': 10})
[tools] fulltext_search({'query': 'subsidiaries organized under the laws of', 'company_id': '3M', 'year': 2024, 'k': 10}) -> 10 chunk(s)


[grade-retrieval] sufficient=True


[answer] attempt #1


[grade-answer] accepted=True next_action=end
[1 retrieval round(s)] tools called: ['fulltext_search']

A: 3M Company’s 2024 10-K lists the following consolidated subsidiary companies and their jurisdictions of organization (doc_id: **3M/3M_2024_10K.pdf**, p. 169):

- **3M Financial Management Company** — Delaware  
- **3M Innovative Properties Company** — Delaware  
- **3M Interamerica LLC** — Delaware  
- **Aearo Technologies LLC** — Delaware  
- **3M Chemical Operations LLC** — Delaware  
- **3M Fall Protection Company** — Delaware  
- **3M Foreign Holding LLC** — Delaware  
- **Scott Technologies, Inc.** — Delaware  
- **D B Industries, LLC** — Minnesota  
- **3M do Brasil Ltda.** — Brazil  
- **3M Belgium BV** — Belgium  
- **3M Canada Company - Compagnie 3M Canada** — Canada  
- **3M China Limited** — China  
- **3M Specialty Materials (Shanghai) Co., Ltd.** — China  
- **3M France S.A.S.** — France  
- **3M Deutschland GmbH** — Germany  
- **3M Hong Kong Limited** — Hong Kong  
-

In [13]:
print("===== WITH the new tools =====")
result_with_q1 = ask(agent_with, Q1)

===== WITH the new tools =====


[strategy] iteration 1: 1 tool call(s) planned
    - get_recognised_entities({'doc_id': '3M/3M_2024_10K.pdf', 'entity_type': 'Company'})
[tools] get_recognised_entities({'doc_id': '3M/3M_2024_10K.pdf', 'entity_type': 'Company'}) -> 36 chunk(s)


[grade-retrieval] sufficient=False
    feedback: The missing piece is the jurisdiction/country of organization for each listed subsidiary. The current chunks do not include that information. The best next step is to use get_document_pages on doc_id 3M/3M_2024_10K.pdf around the page(s) containing the subsidiaries table, because the raw 10-K page text may include the full column(s) with jurisdiction. If that still doesn’t surface the jurisdictions, try fulltext_search or semantic_search on 3M/3M_2024_10K.pdf with terms like 'incorporated', 'organized under the laws of', 'subsidiaries', or specific subsidiary names to locate the table/header that includes jurisdiction entries.


[strategy] iteration 2: 1 tool call(s) planned
    - fulltext_search({'query': 'subsidiaries organized under the laws of incorporated jurisdiction table', 'company_id': '3M', 'year': 2024, 'k': 5})
[tools] fulltext_search({'query': 'subsidiaries organized under the laws of incorporated jurisdiction table', 'company_id': '3M', 'year': 2024, 'k': 5}) -> 5 chunk(s)


[grade-retrieval] sufficient=True


[answer] attempt #1


[grade-answer] accepted=True next_action=end
[2 retrieval round(s)] tools called: ['get_recognised_entities', 'fulltext_search']

A: According to 3M’s 2024 10-K, the consolidated subsidiaries listed are organized under the following jurisdictions (all from doc_id **3M/3M_2024_10K.pdf**):

- **3M Financial Management Company** — Delaware  
- **3M Innovative Properties Company** — Delaware  
- **3M Interamerica LLC** — Delaware  
- **Aearo Technologies LLC** — Delaware  
- **3M Chemical Operations LLC** — Delaware  
- **3M Fall Protection Company** — Delaware  
- **3M Foreign Holding LLC** — Delaware  
- **Scott Technologies, Inc.** — Delaware  
- **D B Industries, LLC** — Minnesota  
- **3M do Brasil Ltda.** — Brazil  
- **3M Belgium BV** — Belgium  
- **3M Canada Company - Compagnie 3M Canada** — Canada  
- **3M China Limited** — China  
- **3M Specialty Materials (Shanghai) Co., Ltd.** — China  
- **3M France S.A.S.** — France  
- **3M Deutschland GmbH** — Germany  
- **3M Hong Kong L

**What actually happened this run.** The *without* agent's single `fulltext_search` call
retrieved the full subsidiaries table in one round and answered it directly and completely —
every company, every jurisdiction, no hedging.

The *with* agent took an extra round to get there, and the extra round is worth reading closely.
Its first call, `get_recognised_entities`, returned all 35 `Company` entities — but section 6
above shows why that wasn't enough: this run's extraction wrote `Location` entities for every
country in the table (see section 8), but no relationship connecting any subsidiary to its
jurisdiction, only `SUBSIDIARY_OF` edges back to the parent. The retrieval-grading step caught
that gap on its own — its feedback reads "The current chunks do not include [jurisdiction]
information" — and fell back to a second-round `fulltext_search`, which supplied it. The final
answer is complete and correct, same as the *without* agent's.

That's actually a different, and arguably better, demonstration than the one this section
originally set out to make: the structured tool alone wasn't sufficient this run, but the
retrieval-grading loop's job — noticing insufficiency and falling back to another tool rather
than accepting an incomplete answer — worked exactly as designed. The one extra round is the
visible cost of an extraction pass that captured the entities but not the relationship tying each
one to its jurisdiction.

### 7b. "How many, and across how many countries?"

A harder question on purpose: it needs an aggregate (count non-US subsidiaries, count *distinct*
countries) over the same subsidiaries table, not just a transcription of it. Computing the ground
truth directly against the graph built in section 6 (the subsidiaries query above, cross-checked
against the *without*-tools agent's own transcription in 7a) gives **35 total subsidiaries, 26
organized outside the US (9 under Delaware/Minnesota law), across 17 distinct non-US countries**
for this run. Extraction is LLM-driven, so a different run over the same chunk can land on
slightly different counts — treat the numbers here as illustrative rather than something to
expect exactly.

In [14]:
Q2 = (
    "Per 3M's 2024 10-K (doc_id 3M/3M_2024_10K.pdf), how many of 3M's consolidated "
    "subsidiaries are organized outside the United States, and across how many distinct "
    "countries?"
)

print("===== WITHOUT the new tools =====")
result_without_q2 = ask(agent_without, Q2)

===== WITHOUT the new tools =====


[strategy] iteration 1: 1 tool call(s) planned
    - fulltext_search({'query': 'consolidated subsidiaries organized outside the United States countries', 'k': 5, 'company_id': '3M', 'year': 2024})
[tools] fulltext_search({'query': 'consolidated subsidiaries organized outside the United States countries', 'k': 5, 'company_id': '3M', 'year': 2024}) -> 5 chunk(s)


[grade-retrieval] sufficient=True


[answer] attempt #1


[grade-answer] accepted=True next_action=end
[1 retrieval round(s)] tools called: ['fulltext_search']

A: Per 3M’s 2024 10-K, **29 consolidated subsidiaries** are organized outside the United States, across **17 distinct countries/jurisdictions**. [doc_id=3M/3M_2024_10K.pdf]


In [15]:
print("===== WITH the new tools =====")
result_with_q2 = ask(agent_with, Q2)

===== WITH the new tools =====


[strategy] iteration 1: 1 tool call(s) planned
    - get_recognised_entities({'doc_id': '3M/3M_2024_10K.pdf', 'entity_type': 'Company'})
[tools] get_recognised_entities({'doc_id': '3M/3M_2024_10K.pdf', 'entity_type': 'Company'}) -> 36 chunk(s)


[grade-retrieval] sufficient=True


[answer] attempt #1


[grade-answer] accepted=True next_action=end
[1 retrieval round(s)] tools called: ['get_recognised_entities']

A: Per 3M’s 2024 10-K, there are **27 consolidated subsidiaries organized outside the United States** and they span **15 distinct countries**. This count excludes the U.S.-organized subsidiaries listed in the filing and excludes 3M Company itself.  
Source: **3M/3M_2024_10K.pdf**


**What actually happened this run.** Neither agent gets the exact count right, and for the same
underlying reason: `MODULE_4_TOOLS` doesn't include a Cypher aggregation tool (that's Module 6's
`query_graph`), so *both* agents are still asking an LLM to count rows and distinct values by
eye — one over retrieved chunk text, the other over a flat list of extracted entity names. The
*without* agent answers **29 organized outside the US, across 17 countries** — the country count
is exactly right, the subsidiary count is off by 3 (ground truth: 26). The *with* agent answers
**27 outside the US, across 15 countries** — closer on the subsidiary count, further off on
countries. Both are plausible-looking wrong answers rather than obviously-wrong ones, which is
the more concerning failure mode of the two: an aggregate that sounds right without actually
being right.

The lesson this run supports: structure changed *what* the with-tools agent got wrong more than
it fixed *whether* the count was right. Neither agent's tools include anything that counts and
dedupes the way Cypher would (`MATCH ... RETURN count(DISTINCT ...)`); until one does, both paths
depend on the LLM doing arithmetic over a list it's holding in context, which this run shows is
unreliable regardless of whether that list came from raw text or structured extraction.

### 7c. The Pattern

This run makes a narrower point than the one this section originally set out to demonstrate. On
the enumeration question (7a), both agents eventually landed on the complete, correct answer —
the *with*-tools agent needed a second round because section 6's extraction captured the
subsidiary entities but not the relationship connecting each one to its jurisdiction, and the
retrieval-grading step correctly noticed the gap and fell back to a text search rather than
accepting an incomplete structured answer. On the aggregate question (7b), neither agent got the
exact count right, because neither has a tool that actually counts — both are an LLM eyeballing
a list either way, structured or not. The real payoff of `RecognisedEntity`/`RELATED_TO` —
reliable, exact aggregation without depending on an LLM's arithmetic — needs two things this run
didn't have together: an extraction pass that captures *every* relevant relationship (not just
the easiest one, `SUBSIDIARY_OF`), and a tool that lets the agent hand counting off to Cypher
instead of doing it itself (Module 6's `query_graph`). Neither gap is unique to this run or this
model; they're the two concrete things worth fixing before this comparison can make the clean
case it was designed to make.

Four chunks, one consistent pipeline: in this run, the subsidiaries table alone produced 35
`Company` entities correctly `SUBSIDIARY_OF` the parent, plus 18 `Location` entities (17 foreign
countries and Delaware) — a real improvement over an earlier run of this same notebook, which
produced zero `Location` entities for this chunk. But look closely at what's still missing: none
of those `Location` entities have an `OPERATES_IN` (or any other) relationship connecting them to
the specific subsidiary organized there. The entities got extracted; the fact tying a company to
its jurisdiction did not — `extract_relationships` only wrote `SUBSIDIARY_OF` edges for this
chunk, 35 of them, and nothing else. That's why section 7a's *with*-tools agent needed a second
retrieval round: `get_recognised_entities` alone can list companies and can list locations, but
can't answer "which jurisdiction is *this* company in" without a relationship between them. A
different run over the same chunk can and does land on a different count, as this notebook has
said throughout — extraction completeness, not just type accuracy, remains genuinely
non-deterministic. Elsewhere, though, structurally: no stray types, no invented relationship
verbs, no relationship pointing at something that was never confirmed to exist (see the
`dropped 1 relationship(s)` log line in section 4 for the backstop catching exactly that). The
PFAS and MD&A chunks are messier (dense, narrative text always will be), but every entity and
relationship in the graph is at least spelled from the same fixed vocabulary and traceable to the
chunk it came from.

What's still true of everything just written: "3M" and "3M Company" are two different
`RecognisedEntity` nodes, and nothing here reconciles them with the curated `Company` nodes from
Module 1/3, even though they obviously refer to the same company. That's Module 5's entity
resolution — this module's job was only to get from unstructured text to a consistent,
queryable, honestly-labeled *first draft* of structure. Run `scripts/run_extraction.py` to
process the rest of the corpus once you're ready to move on.